# Решения: Практика: минимум MSE и итог модуля

**Для преподавателя.** Ниже по разделам разобраны все задачи `lesson.ipynb` и `homework.ipynb`. Не выдавать до сдачи.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_data = np.array([2.2, 3.9, 6.1, 8.0, 10.2])

def predict(w):
    return w * x_data

def mse_for_w(w):
    return float(np.mean((predict(w) - y_data) ** 2))

def derivative_central(fun, x, h=1e-4):
    return float((fun(x + h) - fun(x - h)) / (2 * h))


## Урок. 1. Рельеф MSE по параметру w

In [ ]:
w_grid = np.linspace(0.0, 4.0, 161)
loss_grid = [mse_for_w(w) for w in w_grid]
grid_best_w = float(w_grid[int(np.argmin(loss_grid))])
assert 0 < grid_best_w < 4
print(grid_best_w, min(loss_grid))


## Урок. 2. Знак производной MSE

In [ ]:
probe_w = [0.5, 1.5, 2.5, 3.5]
gradients = [derivative_central(mse_for_w, w) for w in probe_w]
assert gradients[0] < 0 < gradients[-1]
print(list(zip(probe_w, gradients)))


## Урок. 3. Спуск по параметру модели

In [ ]:
def fit_w(start_w, eta=0.05, steps=40):
    w = float(start_w)
    w_path, losses = [w], [mse_for_w(w)]
    for _ in range(steps):
        w -= eta * derivative_central(mse_for_w, w)
        w_path.append(float(w))
        losses.append(mse_for_w(w))
    return w_path, losses

w_path, loss_path = fit_w(0.2)
assert len(w_path) == len(loss_path) == 41 and loss_path[-1] < loss_path[0]
print(w_path[-1], loss_path[-1])


## Урок. 4. Сравнение eta по траектории

In [ ]:
eta_values = [0.01, 0.05, 0.2]
runs = [fit_w(0.2, eta=eta, steps=30) for eta in eta_values]
eta_final_loss = [losses[-1] for _, losses in runs]
eta_status = [
    "stable" if np.all(np.isfinite(losses)) and np.all(np.diff(losses) <= 1e-9) else "unstable"
    for _, losses in runs
]
assert "unstable" in eta_status
print(list(zip(eta_values, eta_final_loss, eta_status)))


## Урок. 5. Разные старты — один минимум

In [ ]:
starts = [0.2, 1.0, 3.0, 4.8]
runs_by_start = [fit_w(start, eta=0.05, steps=40) for start in starts]
end_w = [path[-1] for path, _ in runs_by_start]
end_loss = [losses[-1] for _, losses in runs_by_start]
assert max(end_w) - min(end_w) < 0.02
print(list(zip(starts, end_w, end_loss)))


## Урок. 6. Визуальная проверка модели

In [ ]:
best_w = end_w[0]
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_data, y_data, label="data")
ax.plot(x_data, best_w * x_data, color="crimson", label=f"w={best_w:.3f}")
ax.set(xlabel="x", ylabel="y", title="Модель после минимизации MSE")
ax.legend()
assert len(ax.lines) >= 1 and len(ax.collections) >= 1
plt.show()


## Урок. 7. Сверка с точным минимумом

In [ ]:
exact_w = float(np.sum(x_data * y_data) / np.sum(x_data ** 2))
gap = float(best_w - exact_w)
assert abs(gap) < 0.02
print(exact_w, best_w, gap)


## Урок. 8. Самостоятельно: паспорт запуска

In [ ]:
RUN_REPORT = (
    f"Модель y=w*x обучалась на пяти парах наблюдений. Из старта w=0.2 за 40 шагов при "
    f"eta=0.05 получено w={best_w:.4f} и MSE={mse_for_w(best_w):.4f}. Loss убывал на каждом "
    f"шаге; четыре разных старта пришли к значениям w с разбросом меньше 0.02. Точный минимум "
    f"для этой специальной модели равен {exact_w:.4f}, расхождение с численным ответом {gap:.4g}. "
    "Вывод относится к выпуклому 1D-рельефу; на неровной или многомерной функции одних этих "
    "проверок недостаточно."
)
BRIDGE_NOTE = (
    "В 9 классе параметров станет несколько, а направление будет задаваться набором частных "
    "производных. Из этого модуля переносится инженерный цикл: выбрать шаг, записать траекторию, "
    "проверить конечность и снижение loss, сравнить старты и явно назвать границы вывода."
)
READY = True
assert len(RUN_REPORT) >= 320 and len(BRIDGE_NOTE) >= 180 and READY
print(RUN_REPORT)
print(BRIDGE_NOTE)


## ДЗ. Данные и функции

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_data = np.array([2.2, 3.9, 6.1, 8.0, 10.2])

def predict(w):
    return w * x_data

def mse_for_w(w):
    return float(np.mean((predict(w) - y_data) ** 2))

def derivative_central(fun, x, h=1e-4):
    return float((fun(x + h) - fun(x - h)) / (2 * h))

def fit_w(start_w, eta=0.05, steps=40):
    w = float(start_w)
    w_path, losses = [w], [mse_for_w(w)]
    for _ in range(steps):
        w -= eta * derivative_central(mse_for_w, w)
        w_path.append(float(w))
        losses.append(mse_for_w(w))
    return w_path, losses


## ДЗ. Закрепление: другой старт

In [ ]:
path, losses = fit_w(4.8, eta=0.05, steps=40)
final_w = path[-1]
final_loss = losses[-1]
assert final_loss < losses[0]
print(final_w, final_loss)


## ДЗ. База: автоматическая проверка запуска

In [ ]:
def run_is_stable(losses):
    values = np.asarray(losses, dtype=float)
    return bool(np.all(np.isfinite(values)) and np.all(np.diff(values) <= 1e-9))

stable = run_is_stable(losses)
_, bad_losses = fit_w(0.2, eta=0.2, steps=20)
bad_stable = run_is_stable(bad_losses)
assert stable is True and bad_stable is False
print(stable, bad_stable)


## ДЗ. Углубление: модель со свободным членом

In [ ]:
def mse_for_b(b, fixed_w=2.0):
    predictions = fixed_w * x_data + b
    return float(np.mean((predictions - y_data) ** 2))

def fit_b(start_b, eta=0.05, steps=40):
    b = float(start_b)
    path, losses = [b], [mse_for_b(b)]
    for _ in range(steps):
        b -= eta * derivative_central(mse_for_b, b)
        path.append(float(b))
        losses.append(mse_for_b(b))
    return path, losses

b_path, b_losses = fit_b(-2.0)
assert b_losses[-1] < b_losses[0]
print(b_path[-1], b_losses[-1])


## ДЗ. Вызов: итоговый отчёт полигона

In [ ]:
FINAL_REPORT = (
    "Задача. Мы подбирали один параметр w модели y=w*x по минимуму MSE на пяти наблюдениях.\n\n"
    "Метод. Производную MSE оценивали центральной разностью и обновляли w правилом "
    "w = w - eta * gradient, сохраняя всю траекторию.\n\n"
    f"Свидетельства. При eta=0.05 и старте 4.8 финальный w={final_w:.4f}, "
    f"MSE={final_loss:.4f}; loss снижался монотонно. При eta=0.2 проверка пометила запуск "
    "нестабильным.\n\n"
    "Ограничение. Один параметр и выпуклый MSE-рельеф не показывают локальные минимумы и "
    "взаимодействие нескольких параметров. Совпадение разных стартов здесь ожидаемо и не "
    "является общей гарантией градиентного спуска.\n\n"
    "Следующий вопрос. Как хранить и обновлять направление, если у модели одновременно меняются "
    "w и свободный член b, и как выбирать шаги для параметров разного масштаба?"
)
PERSONAL_NOTE = (
    "За год я научился связывать код с проверяемым утверждением: задавать контракт assert, "
    "сохранять промежуточные результаты, сравнивать метрики и отделять наблюдение от гарантии. "
    "В этом модуле свидетельство — диагностическая функция, которая различает стабильную и "
    "расходящуюся траектории по самим значениям loss."
)
READY = True
assert len(FINAL_REPORT) >= 520 and len(PERSONAL_NOTE) >= 180 and READY
print(FINAL_REPORT)
print(PERSONAL_NOTE)
